# Social choice & voting — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def expected_payoff(A,p,q): return float(np.asarray(p) @ np.asarray(A) @ np.asarray(q))
def best_response_payoffs(A,q): return np.asarray(A) @ np.asarray(q)
print('setup ok')

## Aggregating preferences can create cycles and strategic pressure
Social choice studies the rules for turning many rankings into one decision. These examples compute plurality, Borda, Condorcet majority comparisons, an agenda effect, and a tiny manipulation that shows why voting rules are mechanisms too.

In [ ]:
# 1) Plurality counts only first-place votes
profiles=[('A','B','C'),('A','B','C'),('B','A','C'),('C','B','A'),('C','B','A')]
first={c:sum(p[0]==c for p in profiles) for c in 'ABC'}
plt.figure(figsize=(5,3)); plt.bar(list(first),list(first.values())); plt.title('plurality: A and C tie with 2 first-place votes')
assert first=={'A':2,'B':1,'C':2}

In [ ]:
# 2) Borda uses full rankings, not just first place
scores={c:0 for c in 'ABC'}
for p in profiles:
    for pts,c in zip([2,1,0],p): scores[c]+=pts
plt.figure(figsize=(5,3)); plt.bar(list(scores),list(scores.values())); plt.title('Borda scores: B wins with 6')
assert scores=={'A':5,'B':6,'C':4}

In [ ]:
# 3) Condorcet majority comparisons can cycle
profiles=[('A','B','C'),('B','C','A'),('C','A','B')]
def beats(x,y): return sum(p.index(x)<p.index(y) for p in profiles)
M=np.array([[0,beats('A','B'),beats('A','C')],[beats('B','A'),0,beats('B','C')],[beats('C','A'),beats('C','B'),0]])
plt.figure(figsize=(4,3)); plt.imshow(M,cmap='Purples',vmin=0,vmax=3); plt.xticks([0,1,2],list('ABC')); plt.yticks([0,1,2],list('ABC')); plt.title('pairwise majority counts')
assert beats('A','B')==2 and beats('B','C')==2 and beats('C','A')==2

In [ ]:
# 4) Agenda control: the order of pairwise votes changes the winner
# If A vs B first, A wins, then C beats A. If B vs C first, B wins, then A beats B.
agenda_winners={'(A vs B) then C':'C','(B vs C) then A':'A','(C vs A) then B':'B'}
plt.figure(figsize=(6,3)); plt.bar(list(agenda_winners.keys()),[{'A':0,'B':1,'C':2}[w] for w in agenda_winners.values()]); plt.yticks([0,1,2],['A','B','C']); plt.title('same voters, different agenda winners')
assert set(agenda_winners.values())==set('ABC')

In [ ]:
# 5) Strategic voting under plurality: one voter can change the winner
truth=[('A','B','C'),('B','C','A'),('C','B','A')]
def plurality(prof):
    counts={c:sum(p[0]==c for p in prof) for c in 'ABC'}; return counts, max(counts,key=counts.get)
counts_true,w_true=plurality(truth)
strategic=[('B','A','C'),('B','C','A'),('C','B','A')]
counts_strat,w_strat=plurality(strategic)
plt.figure(figsize=(6,3)); plt.bar(['A truthful','B truthful','C truthful'],[counts_true[c] for c in 'ABC'],alpha=.6,label='truth'); plt.bar(['A strategic','B strategic','C strategic'],[counts_strat[c] for c in 'ABC'],alpha=.6,label='strategic'); plt.legend(); plt.title('changing one ballot changes plurality winner')
assert w_true=='A' and w_strat=='B'